In [1]:
import csv
import os


def reconcile_inventory(
    inventory_path="inventory.csv",
    movements_path="movements.csv",
    updated_path="updated_inventory.csv",
    rejected_path="rejected_movements.csv"
):
    if not os.path.exists(inventory_path):
        raise FileNotFoundError(f"Inventory file nahi mili: {inventory_path}")
    if not os.path.exists(movements_path):
        raise FileNotFoundError(f"Movements file nahi mili: {movements_path}")

    inventory = {}
    sku_order = []

    # Step 1: inventory.csv read karein
    with open(inventory_path, "r", newline="", encoding="utf-8") as inv_file:
        reader = csv.DictReader(inv_file)
        if not reader.fieldnames or not {"sku", "name", "quantity"}.issubset(set(reader.fieldnames)):
            raise ValueError("inventory.csv mein sku, name, aur quantity columns hone chahiye")

        for row in reader:
            sku = (row.get("sku") or "").strip()
            name = (row.get("name") or "").strip()
            qty_raw = (row.get("quantity") or "").strip()

            if not sku:
                continue

            try:
                qty = int(qty_raw)
                if qty < 0:
                    raise ValueError("Initial quantity negative nahi ho sakti")
            except ValueError:
                raise ValueError(f"Invalid quantity inventory mein SKU '{sku}' ke liye: {qty_raw}")

            if sku in inventory:
                raise ValueError(f"Duplicate SKU inventory mein mila: {sku}")

            inventory[sku] = {"sku": sku, "name": name, "quantity": qty}
            sku_order.append(sku)

    rejected_rows = []

    # Step 2: movements.csv read karke row by row apply karein
    with open(movements_path, "r", newline="", encoding="utf-8") as mov_file:
        reader = csv.DictReader(mov_file)
        if not reader.fieldnames or not {"sku", "type", "units"}.issubset(set(reader.fieldnames)):
            raise ValueError("movements.csv mein sku, type, aur units columns hone chahiye")

        for row in reader:
            sku = (row.get("sku") or "").strip()
            mov_type = (row.get("type") or "").strip().upper()
            units_raw = (row.get("units") or "").strip()

            # Check 1: Unknown SKU
            if sku not in inventory:
                rejected_rows.append({
                    "sku": sku,
                    "type": row.get("type", ""),
                    "units": units_raw,
                    "reason": "Unknown SKU"
                })
                continue

            # Check 2: Movement type IN ya OUT honi chahiye
            if mov_type not in ("IN", "OUT"):
                rejected_rows.append({
                    "sku": sku,
                    "type": row.get("type", ""),
                    "units": units_raw,
                    "reason": "Invalid movement type"
                })
                continue

            # Check 3: Units valid integer aur positive (> 0) hone chahiye
            try:
                units = int(units_raw)
                if units <= 0:
                    raise ValueError()
            except ValueError:
                rejected_rows.append({
                    "sku": sku,
                    "type": mov_type,
                    "units": units_raw,
                    "reason": "Units must be a positive integer"
                })
                continue

            # Check 4: Stock update aur negative stock validation
            current_qty = inventory[sku]["quantity"]
            if mov_type == "IN":
                inventory[sku]["quantity"] = current_qty + units
            elif mov_type == "OUT":
                if units > current_qty:
                    rejected_rows.append({
                        "sku": sku,
                        "type": mov_type,
                        "units": units,
                        "reason": "Insufficient stock"
                    })
                    continue
                inventory[sku]["quantity"] = current_qty - units

    # Step 3: updated_inventory.csv write karein
    with open(updated_path, "w", newline="", encoding="utf-8") as out_inv:
        writer = csv.DictWriter(out_inv, fieldnames=["sku", "name", "quantity"])
        writer.writeheader()
        for sku in sku_order:
            writer.writerow(inventory[sku])

    # Step 4: rejected_movements.csv write karein
    with open(rejected_path, "w", newline="", encoding="utf-8") as out_rej:
        writer = csv.DictWriter(out_rej, fieldnames=["sku", "type", "units", "reason"])
        writer.writeheader()
        for item in rejected_rows:
            writer.writerow(item)

    return inventory, rejected_rows


# Sample data generation & testing
if __name__ == "__main__":
    with open("inventory.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["sku", "name", "quantity"])
        writer.writerow(["P01", "Wireless Mouse", 10])
        writer.writerow(["P02", "Mechanical Keyboard", 5])

    with open("movements.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["sku", "type", "units"])
        writer.writerow(["P01", "OUT", 4])     # Valid -> P01 = 6
        writer.writerow(["P01", "OUT", 9])     # Rejected -> Insufficient stock
        writer.writerow(["P01", "IN", 2])      # Valid -> P01 = 8
        writer.writerow(["P99", "IN", 5])      # Rejected -> Unknown SKU
        writer.writerow(["P02", "HOLD", 2])    # Rejected -> Invalid movement type
        writer.writerow(["P02", "OUT", 0])     # Rejected -> Non-positive units

    updated, rejected = reconcile_inventory()
    print("Updated Inventory:", updated)
    print("Rejected Movements:", rejected)
    

Updated Inventory: {'P01': {'sku': 'P01', 'name': 'Wireless Mouse', 'quantity': 8}, 'P02': {'sku': 'P02', 'name': 'Mechanical Keyboard', 'quantity': 5}}
Rejected Movements: [{'sku': 'P01', 'type': 'OUT', 'units': 9, 'reason': 'Insufficient stock'}, {'sku': 'P99', 'type': 'IN', 'units': '5', 'reason': 'Unknown SKU'}, {'sku': 'P02', 'type': 'HOLD', 'units': '2', 'reason': 'Invalid movement type'}, {'sku': 'P02', 'type': 'OUT', 'units': '0', 'reason': 'Units must be a positive integer'}]
